# NB10 — Portfolio Construction
**Machine Learning in Finance · Week 10**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB10_portfolio_construction.ipynb)

**Finance question:** I have a ranked list of stocks. How much of each should I buy?

**Key idea:** The model ranks; the **portfolio** decides how much money goes where. Textbook "optimal" weights are **fragile** when the inputs are noisy, so practitioners use simple, robust schemes: equal weight, rank-weighted, inverse volatility, and caps on how much sits in one cluster.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: the efficient frontier, as a picture
Ten familiar stocks, 2015–2019. Each blue dot is one random mix of them (long only, weights add to 100%),
placed at its **risk** (volatility) and **return**. Gold dots are the stocks on their own; the white star
is equal weight. The upper-left edge of the cloud is the *efficient frontier*: the most return for each
level of risk.

In [ ]:
print(mlfin.PORTFOLIO_STOCKS)
mlfin.plot_frontier(start="2015", end="2019")

*What to look for:* most mixes are less risky than most single stocks (diversification), and the
frontier bends: you can lower risk a lot before giving up much return.

## Step 2: the "optimal" portfolio, estimated twice
Mean-variance optimization picks the weights with the best return per unit of risk (the highest
**Sharpe ratio**), treating past averages and past co-movements as if they were the truth. Here it may
also **short** (bet against) stocks, as in the textbook. Then we estimate it again with **one year** of
data removed. If the answer were reliable, the two sets of weights would look alike.

In [ ]:
# 🟡 YOUR TURN
DROP_YEAR = 2016   # ← remove one year of the 2015–2019 data: try 2015, 2017, 2018, 2019

weights = mlfin.fragility_experiment(start="2015", end="2019", drop_year=DROP_YEAR)
weights

*What to look for:* weights far above 100% or below 0% (big short bets), and how much they change
when one year out of five is removed.

## Step 3: did "optimal" pay off?
Hold each portfolio from 2020 to 2025 (the years after the estimate) and compare it with plain equal weight.

In [ ]:
mlfin.out_of_sample_check(est_start="2015", est_end="2019", test_start="2020", test_end="2025")

## Step 4: four practical ways to weight the same picks
Now the course model from Weeks 6–7 picks 30 stocks every quarter. Same picks, four weighting schemes:

* **Equal weight**: the same amount in each.
* **Rank-weighted**: more in the stocks the model likes most.
* **Inverse volatility**: less in jumpy stocks, more in calm ones.
* **Cluster-balanced**: equal weight, but at most 4 stocks from any one cluster of stocks that move
  together (Week 9). Clusters are re-estimated each year from the previous five years only.

(Four backtests: about 20 seconds.)

In [ ]:
panel = mlfin.make_features()
model = mlfin.default_model()
table, results = mlfin.compare_weightings(panel, model, n_long=30, rebalance="Q", cost_bps=20)
table

*What to look for:* the return column barely changes; the **risk** columns (volatility, worst
drawdown) do. Weighting is mostly a risk decision.

## Step 5: build your fund
Five dials. Change them, run, and compare with the table above.

In [ ]:
# 🟡 YOUR TURN
N_LONG = 30             # ← how many stocks: 10, 30, 100
WEIGHTING = "cluster"   # ← "equal", "rank", "inverse_vol" or "cluster"
REBALANCE = "Q"         # ← "M" monthly, "Q" quarterly, "A" yearly
MAX_WEIGHT = None       # ← cap on any one stock, e.g. 0.05 for 5% (None = no cap)
MAX_PER_CLUSTER = None  # ← at most this many stocks from one cluster, e.g. 3 (None = no cap; "cluster" uses 4)

fund = mlfin.walk_forward_backtest(panel, model, n_long=N_LONG, weighting=WEIGHTING, rebalance=REBALANCE,
                                   max_weight=MAX_WEIGHT, max_per_cluster=MAX_PER_CLUSTER, verbose=False)
mlfin.plot_cumulative(fund)
fund["stats"]

## Step 6: how often should this fund trade?
Your weighting scheme, rebalanced monthly, quarterly and yearly. (Three backtests: about 15 seconds.)

In [ ]:
mlfin.rebalance_experiment(panel, model, freqs=["M", "Q", "A"], n_long=N_LONG, weighting=WEIGHTING)

## Step 7: where does the money sit?
The fund's most recent picks, grouped into clusters of stocks that move together.

In [ ]:
mlfin.portfolio_concentration(fund)

## ⭐ GOING FURTHER (optional)
1. Long-only "optimal" weights: `mlfin.mean_variance_weights(start="2015", end="2019", long_only=True)`.
   How many stocks get zero? Does the long-only version survive Step 3 better than the shorting one?
2. Use ten stocks of your choice: `mlfin.plot_frontier(tickers=[...], start="2015", end="2019")`.
3. Ask an AI assistant: *"Why do portfolio managers rarely use unconstrained mean-variance weights?"*
   Check its answer against Steps 2–3.

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*